# ML Feature Engineering & Anomaly Prediction

This notebook demonstrates how the **Gold ML features table** directly
supports machine learning workflows. We:

1. Explore the pre-computed feature vector from the Gold layer
2. Analyze feature correlations and distributions
3. Train a Random Forest classifier to predict anomalous device windows
4. Evaluate model performance (precision, recall, ROC)
5. Extract feature importance to explain which signals drive anomaly risk

The key insight: the pipeline's feature engineering (z-scores, quality scores,
per-sensor anomaly counts) is done *once* in the Silver/Gold layers and
reused across any number of downstream models.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

## 1. Load the Gold ML Features Table

This table was computed by `compute_ml_features()` in the Gold layer.
Each row represents one `(device_id, time_window)` with 33+ pre-computed features.

In [ ]:
ml_path = str(Path(config["paths"]["delta_gold"]) / "ml_features")
ml_df = spark.read.format("delta").load(ml_path)

print(f"Shape: {ml_df.count()} rows x {len(ml_df.columns)} columns")
print(f"\nColumns:")
for c in ml_df.columns:
    print(f"  {c}")

In [ ]:
features_pdf = ml_df.toPandas()

# Also load device health for enrichment
health_path = str(Path(config["paths"]["delta_gold"]) / "device_health")
health_pdf = spark.read.format("delta").load(health_path).toPandas()

spark.stop()

features_pdf.head()

## 2. Feature Exploration

Understand the distribution and relationships among features
before building a model.

In [ ]:
# Numeric features only (exclude IDs, timestamps)
feature_cols = [
    c for c in features_pdf.columns
    if c not in ("device_id", "window_start", "window_end", "_aggregated_at")
]

desc = features_pdf[feature_cols].describe().T
desc[["mean", "std", "min", "max"]]

In [ ]:
# Feature correlation heatmap
corr = features_pdf[feature_cols].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns, y=corr.index,
    colorscale="RdBu_r", zmin=-1, zmax=1,
    text=np.round(corr.values, 2), texttemplate="%{text}",
    textfont={"size": 7},
))
fig.update_layout(
    title="Feature Correlation Matrix",
    template="plotly_white", height=700, width=800,
    xaxis_tickangle=-45,
)
fig.show()

### Key Feature Distributions

Compare sensor ranges and anomaly rates across device windows.

In [ ]:
key_features = ["temp_range", "hum_range", "pres_range",
                "anomaly_rate", "quality_mean", "battery_mean"]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f.replace("_", " ").title() for f in key_features],
)

for i, feat in enumerate(key_features):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    vals = features_pdf[feat].dropna()
    fig.add_trace(
        go.Histogram(x=vals, nbinsx=30, marker_color="#3498db", showlegend=False),
        row=row, col=col_idx,
    )

fig.update_layout(
    title="Key Feature Distributions",
    template="plotly_white", height=500,
)
fig.show()

## 3. Define the Prediction Target

We define a binary classification task:

> **Predict whether a device-window has an anomaly rate > 0**
> (i.e., at least one anomalous reading in that time window)

This is a practical framing: given the aggregated features for a device
in a time window, can we predict if it experienced any anomalies?

In [ ]:
# Target: device-window had at least one anomaly
features_pdf["is_anomalous_window"] = (features_pdf["anomaly_rate"] > 0).astype(int)

target_dist = features_pdf["is_anomalous_window"].value_counts()
print("Target distribution:")
print(f"  Clean windows:     {target_dist.get(0, 0)}")
print(f"  Anomalous windows: {target_dist.get(1, 0)}")
print(f"  Anomaly prevalence: {features_pdf['is_anomalous_window'].mean()*100:.1f}%")

In [ ]:
# Select model features (exclude target-leaking columns)
# anomaly_count, anomaly_rate, and per-type counts directly encode the target,
# so we exclude them. The model must learn from sensor patterns only.
leak_cols = {
    "anomaly_count", "anomaly_rate", "is_anomalous_window",
    "temp_anomaly_count", "hum_anomaly_count", "pres_anomaly_count",
}
model_features = [
    c for c in feature_cols
    if c not in leak_cols
]

print(f"Model features ({len(model_features)}):")
for f in model_features:
    print(f"  {f}")

## 4. Train an Anomaly Classification Model

We use a Random Forest because it:
- Handles mixed feature scales without normalization
- Provides feature importance rankings natively
- Is robust to missing values and outliers
- Serves as a strong baseline that's easy to explain

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)

X = features_pdf[model_features].fillna(0)
y = features_pdf["is_anomalous_window"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y,
)

print(f"Train: {len(X_train)} samples ({y_train.mean()*100:.1f}% positive)")
print(f"Test:  {len(X_test)} samples ({y_test.mean()*100:.1f}% positive)")

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced",
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Clean", "Anomalous"]))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ["Clean", "Anomalous"]

fig = go.Figure(go.Heatmap(
    z=cm, x=labels, y=labels,
    text=cm, texttemplate="%{text}",
    colorscale="Blues",
))
fig.update_layout(
    title="Confusion Matrix",
    xaxis_title="Predicted", yaxis_title="Actual",
    template="plotly_white", height=400, width=450,
)
fig.show()

### ROC Curve

In [ ]:
if len(np.unique(y_test)) > 1:
    auc = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"ROC (AUC = {auc:.3f})",
        line=dict(color="#3498db", width=2),
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines",
        name="Random Baseline",
        line=dict(color="gray", dash="dash"),
    ))
    fig.update_layout(
        title=f"ROC Curve (AUC = {auc:.3f})",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        template="plotly_white", height=450, width=550,
    )
    fig.show()
else:
    print("Only one class present in test set — ROC not applicable.")

## 5. Feature Importance

Which features does the model rely on most? This tells us what sensor
patterns are the strongest predictors of anomalous windows.

In [ ]:
importances = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=True)

top_n = importances.tail(15)

fig = go.Figure(go.Bar(
    y=top_n["feature"],
    x=top_n["importance"],
    orientation="h",
    marker_color="#3498db",
    text=top_n["importance"].round(3),
    textposition="outside",
))
fig.update_layout(
    title="Top 15 Feature Importances (Random Forest)",
    xaxis_title="Importance",
    template="plotly_white", height=500,
)
fig.show()

### Feature Category Contribution

Group importances by category to see which *types* of features matter most.

In [ ]:
def categorize(feat):
    if "zscore" in feat:
        return "Z-Score Features"
    elif "quality" in feat:
        return "Quality Features"
    elif "battery" in feat:
        return "Battery Features"
    elif "temp" in feat:
        return "Temperature Sensor"
    elif "hum" in feat:
        return "Humidity Sensor"
    elif "pres" in feat:
        return "Pressure Sensor"
    else:
        return "Other"

importances["category"] = importances["feature"].apply(categorize)
cat_importance = importances.groupby("category")["importance"].sum().sort_values()

fig = go.Figure(go.Bar(
    y=cat_importance.index,
    x=cat_importance.values,
    orientation="h",
    marker_color=["#e74c3c", "#3498db", "#9b59b6", "#f39c12", "#2ecc71", "#1abc9c", "#95a5a6"],
    text=cat_importance.values.round(3),
    textposition="outside",
))
fig.update_layout(
    title="Feature Category Importance",
    xaxis_title="Total Importance",
    template="plotly_white", height=350,
)
fig.show()

## 6. Model Predictions on the Full Dataset

Score all device-windows and compare model predictions against actual anomaly status.

In [ ]:
features_pdf["predicted_prob"] = model.predict_proba(
    features_pdf[model_features].fillna(0)
)[:, 1]
features_pdf["predicted_anomalous"] = (features_pdf["predicted_prob"] > 0.5).astype(int)

results = features_pdf.merge(
    health_pdf[["device_id", "health_score", "risk_tier"]],
    on="device_id", how="left",
)

fig = px.scatter(
    results, x="health_score", y="predicted_prob",
    color="risk_tier",
    color_discrete_map={"healthy": "#2ecc71", "warning": "#f39c12", "critical": "#e74c3c"},
    hover_data=["device_id", "anomaly_rate", "quality_mean"],
    title="Model Anomaly Probability vs Gold Health Score",
    labels={"predicted_prob": "Predicted Anomaly Probability", "health_score": "Health Score"},
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

## 7. Operationalizing the Model

In production, this model would:

1. **Ingest** — Gold layer recomputes ML features on each batch/window
2. **Score** — The trained model predicts anomaly probability for each device-window
3. **Alert** — Devices with `predicted_prob > threshold` trigger operational alerts
4. **Monitor** — Model drift detected by comparing predicted vs actual anomaly rates

### Scoring New Data

Here's how to score a new device-window using the pre-trained model:

In [ ]:
new_window = features_pdf[model_features].iloc[[0]].copy()
new_window["temp_mean"] = 155.0   # abnormally high
new_window["temp_range"] = 130.0  # extreme variation
new_window["temp_zscore_abs_mean"] = 8.0  # high z-score

prob = model.predict_proba(new_window.fillna(0))[0][1]
label = "ANOMALOUS" if prob > 0.5 else "CLEAN"

print(f"Prediction:  {label}")
print(f"Probability: {prob:.3f}")
print(f"\nThis shows how the Gold ML features table serves as a")
print(f"ready-made feature store for real-time or batch scoring.")

## Summary

| Pipeline Stage | ML Contribution |
|---|---|
| **Bronze** | Raw data preserved for reprocessing and backfill |
| **Silver** | Per-record anomaly flags, z-scores, quality scores |
| **Gold** | Pre-aggregated feature vectors, ready for model training |
| **Model** | Random Forest classifier, feature importance, scoring |

The medallion architecture provides a natural separation between
**data engineering** (Bronze/Silver) and **feature engineering** (Gold),
so ML practitioners consume clean, versioned feature tables without
needing to understand the underlying streaming pipeline.